In [106]:
import duckdb
import pandas as pd
import plotly.express as px
import numpy as np

db = duckdb.connect(
    "C:/Users/Decss/Desktop/SQL-övning/Data/retail.db",
    read_only=True
)
db.sql("SET search_path TO mart;")
db.sql("""SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'mart'""")

┌─────────────────┐
│   table_name    │
│     varchar     │
├─────────────────┤
│ customerdim     │
│ productdim      │
│ retail_enriched │
└─────────────────┘

Uppdrag 1 — Customer Segmentation (RFM)
Business context

E-commerce teamet vill förstå vilka kunder som är mest värdefulla.


Fråga

Kan vi segmentera våra kunder baserat på beteende och identifiera de viktigaste kundgrupperna?


Skapa RFM:

Recency
Frequency
Monetary

Segment:

Segment	Beskrivning
Champions	många köp + nyligen
Loyal	många köp
Big Spenders	hög spend
At Risk	köpt länge sedan
New Customers	få köp
Deliverable

En tabell:

| segment | customers | revenue_share |

Insight

Exempel:

12 % av kunder står för 54 % av intäkterna.

Rekommendation
loyalty program
VIP discounts
retention campaigns


    Notering: 
“Orders från staging.top_anomalies med anomaly_type = 'anomaly_customer' har filtrerats bort. Dessa representerar stora manuella justeringar, plattformsavgifter eller ovanligt stora kunder som snedvrider analysen.”

In [107]:
# Antal unika registrerade användare och unika gästorders
reg_n_guest = db.sql("""
SELECT
COUNT(DISTINCT customerid) AS n_customers,
COUNT(DISTINCT CASE WHEN is_missing_customerid =1 THEN invoiceno END) AS n_guest_orders
FROM retail_enriched
""").df()
print(reg_n_guest)

   n_customers  n_guest_orders
0         4372            3710


In [108]:
# Totala intäkter uppdelat på registrerade kunder och gästorders. Exkl anomalikunder
customer_rev = db.sql("""
       SELECT 
       CASE 
        WHEN is_missing_customerid = 0 THEN 'Registered'
        ELSE 'Guest'
       END as customer_type,
       ROUND(SUM(total_price), 2) as revenue
       FROM retail_enriched  
       WHERE total_price > 0 
       AND is_product = 1
       AND (customerid NOT IN (
       SELECT DISTINCT customerid 
       FROM staging.top_anomalies 
       WHERE anomaly_type = 'anomaly_customer'
       ) OR customerid IS NULL)
       GROUP BY customer_type

       UNION ALL

       SELECT 
       'Total' as customer_type,
       ROUND(SUM(total_price), 2) as revenue
       FROM retail_enriched
       WHERE total_price > 0 
       AND is_product = 1
       AND (customerid NOT IN (
       SELECT DISTINCT customerid 
       FROM staging.top_anomalies 
       WHERE anomaly_type = 'anomaly_customer'
       ) OR customerid IS NULL)
       """).df()
print(customer_rev)

  customer_type     revenue
0    Registered  8449765.23
1         Guest  1520635.91
2         Total  9970401.14


In [109]:
# Isolerar Registered och Guest. Plottar sedan i pajgraf.
df_rev_plot = customer_rev[customer_rev['customer_type'] != "Total"]
px.pie(df_rev_plot,
       values='revenue',
       names='customer_type',
       title='Registered customers and Guests share of revenue')

Tidsperiod: 2010-12 to 2011-12 

Unika registrerade användare och unika gästorders.

    Registrerade kunder: 4372
    Gästorders: 3710

Totala intäkter: 10255973

    Registrerade kunder: 84,7% av total
    Gäster: 15,3% av total

In [110]:
# Jämför returmått - Registrerad vs Gäst
customer_guest_return = db.sql("""
       SELECT 
                      
       CASE 
        WHEN is_missing_customerid = 0 THEN 'Registered'
        ELSE 'Guest'
       END as customer_type,
                      
       ROUND(COUNT(DISTINCT CASE WHEN is_return_line = 1 THEN invoiceno END) * 1.0 / NULLIF(COUNT(DISTINCT CASE WHEN is_credit_invoice = 0 THEN invoiceno END), 0), 4) as return_rate_order,
       ROUND(SUM(CASE WHEN total_price < 0 THEN ABS(total_price) END) / NULLIF(SUM(CASE WHEN total_price > 0 THEN total_price END), 0), 4) as return_rate_value


       FROM retail_enriched  
       WHERE is_product = 1
       AND (customerid NOT IN (
       SELECT DISTINCT customerid 
       FROM staging.top_anomalies 
       WHERE anomaly_type = 'anomaly_customer'
       ) OR customerid IS NULL)
       GROUP BY customer_type
                      
       """).df()
print(customer_guest_return)

  customer_type  return_rate_order  return_rate_value
0    Registered             0.1879             0.0283
1         Guest             0.0234             0.0242


In [111]:
# Jämför korgar - Registrerad vs Gäst
customer_guest_basket = db.sql("""
                               WITH basket_items AS (
                               
                                   SELECT 

                                   invoiceno,

                                   CASE 
                                   WHEN is_missing_customerid = 0 THEN 'Registered'
                                   ELSE 'Guest'
                                   END as customer_type,

                                   COUNT(DISTINCT stockcode) as n_items,
                                   SUM(total_price) as basket_value

                                   FROM retail_enriched
                                   WHERE total_price > 0
                                   AND is_product = 1 
                                   AND (customerid NOT IN (
                                   SELECT DISTINCT customerid 
                                   FROM staging.top_anomalies 
                                   WHERE anomaly_type = 'anomaly_customer'
                                   ) OR customerid IS NULL) 
                                   GROUP BY invoiceno, customer_type
                               )

       SELECT 

       customer_type,                
       ROUND(AVG(basket_value), 2) as avg_basket_value,
       ROUND(MEDIAN(basket_value), 2) as median_basket_value,
       ROUND(AVG(n_items), 2) as avg_n_items,
       ROUND(MEDIAN(n_items), 2) as median_n_items

       FROM basket_items
       GROUP BY customer_type

       """).df()
print(customer_guest_basket)


  customer_type  avg_basket_value  median_basket_value  avg_n_items  \
0    Registered            459.00               301.29        20.96   
1         Guest           1106.72               361.15        95.24   

   median_n_items  
0            15.0  
1            38.0  


In [112]:
# Plottar korgvärden - registrerad vs gäst
fig_cgb = px.bar(customer_guest_basket,
       x='customer_type',
       y=['avg_basket_value','median_basket_value'],
       title='Basket Values',
       barmode='group')

fig_cgb.update_layout(
    xaxis_title='Customer Type',
    yaxis_title='Value',
    legend_title='Metric'
)

In [113]:
# Plottar unika varor per korg - registrerad vs gues
fig_cgi = px.bar(customer_guest_basket,
                 x='customer_type',
                 y=['avg_n_items', 'median_n_items'],
                 title='Basket items',
                 barmode='group')
fig_cgi.update_layout(
    xaxis_title='Customer Type',
    yaxis_title='Number of items',
    legend_title='Metric'
)

fig_cgi.show()

    Intäkter drivs av återkommande kunder, inte enstaka köp.

Ett problem i denna jämförelse är att vi antar att varje gästorder är en unik gäst, när det faktiskt skulle kunna vara ett par återkommande kunder som inte är registrerade, men det kan vi inte se. 

Vad vi kan se är däremot att gäster tycks spendera i genomsnitt mer per köp än vad våra registrerade kunder gör. De handlar bredare med betydligt mer variation i korgarna per order, men de handlar också betydligt mer sällan än våra registrerade kunder.

Gästernas totala köp står däremot endast för 15,3% av totala intäkter. Det är de registrerade kundernas återkommande köp som driver våra intäkter. Att konvertera gäster till mer återkommande kunder bör därav ses som en möjlighet att öka intäkter, inte per order, utan totalt sett på lång sikt.

De skillnader som kan observeras i returmåtten är troligtvis kopplade till köpfrekvens snarare än faktisk produktmissnöje. Fler orders innebär fler tillfällen där en enskild vara inte uppfyller förväntningarna eller har pushats fel. Utan data på returanledningar kan det inte bekräftas.

I och med att registrerade kunder står för hela 84,7% av intäkterna så ska vi titta lite närmare på hur de beter sig och hur de skiljer sig åt inom gruppen. 

In [114]:
df_reg_customer = db.sql("""
                         SELECT
                         total_spend,
                         days_since_last_purchase,
                         total_orders
                         FROM customerdim
                         WHERE (customerid NOT IN (
                              SELECT DISTINCT customerid 
                              FROM staging.top_anomalies 
                              WHERE anomaly_type = 'anomaly_customer'
                              ) OR customerid IS NULL)
                         ORDER BY days_since_last_purchase
                         """).df()
df_reg_customer[['total_spend','days_since_last_purchase', 'total_orders']].describe()

,total_spend,days_since_last_purchase,total_orders
count,4335.000000,4336.0,4369.000000
mean,1984.229826,91.988238,4.241245
std,8530.068505,99.960643,7.687636
min,3.750000,0.0,0.000000
25%,306.455000,17.0,1.000000
50%,668.430000,50.0,2.000000
75%,1655.795000,141.0,5.000000
max,280206.020000,373.0,210.000000


In [115]:
# Distribution av data (total_spend, total_orders, days_since_last_purchase)
fig_spend = px.histogram(df_reg_customer,
             x='total_spend',
             title='Total Spend distribution'
             )
fig_spend.update_traces(xbins=dict(size=250))
fig_spend.show()
px.histogram(df_reg_customer,
             x='total_orders',
             title='Total Orders distribution').show()
px.histogram(df_reg_customer,
             x='days_since_last_purchase',
             title='Days since last purchase distribution').show()

In [116]:
# Distribution av (total_spend, total_orders, days_since_last_purchase).
# Hanterar skew och extrema värden genom att Logaritmera total_spend och total_orders
plot_reg_customer = df_reg_customer.copy()
plot_reg_customer['log_spend'] = np.log1p(plot_reg_customer['total_spend'])
plot_reg_customer['log_order'] = np.log1p(plot_reg_customer['total_orders'])

px.histogram(plot_reg_customer,
             x='log_spend',
             title='log Total Spend distribution').show()
px.histogram(plot_reg_customer,
             x='log_order',
             title='log Total Orders distribution').show()
px.histogram(plot_reg_customer,
             x='days_since_last_purchase',
             title='Days since last purchase distribution').show()
# Överväg Jenks Natural Breaks?

In [117]:
db.sql("""SELECT 
    PERCENTILE_CONT(0.33) WITHIN GROUP (ORDER BY total_orders) as freq_low,
    PERCENTILE_CONT(0.67) WITHIN GROUP (ORDER BY total_orders) as freq_high,
    PERCENTILE_CONT(0.33) WITHIN GROUP (ORDER BY total_spend) as spend_low,
    PERCENTILE_CONT(0.67) WITHIN GROUP (ORDER BY total_spend) as spend_high
    FROM mart.customerdim
       """)

┌──────────┬───────────┬────────────────────┬────────────────────┐
│ freq_low │ freq_high │     spend_low      │     spend_high     │
│  double  │  double   │       double       │       double       │
├──────────┼───────────┼────────────────────┼────────────────────┤
│      1.0 │       4.0 │ 384.10100000000006 │ 1212.8021999999999 │
└──────────┴───────────┴────────────────────┴────────────────────┘

Att kunna identifiera kundgrupper som driver värde är centralt i skapandet av riktade insatser. Det avgör inte bara vad för typ av insats vi pushar utan också när och hur ofta.

Genom att titta på kundmått som total_orders, days_since_last_purchase och total_spend kan vi gruppera kunder efter beteende snarare än demografi, och därmed prioritera rätt insats till rätt grupp.

Utifrån hur distributionen för dessa mått ser ut så sätter vi brytpunkter för hur vi ska kategorisera värdena. I days_since_last_purchase går det att urskilja två brytpunkter där rate of change tydligt avtar. Dessa brytpunkter, vid 39 och 89 dagar hjälper oss att definiera de nivåer, low, mid och high, som vi kommer att använda oss av för att dela in kunderna i de olika segmenten.

För days_since_last_purchase blir Low: 0-39, Mid: 40-89, High: 90+

För total_spend och total_orders är valet inte lika självklart då datan i båda måtten är väldigt skeva. På grund av detta så delar vi upp dessa i percentiler, 33e och 67e, vilket sammanfaller väl med valet att använda tre nivåer i days_since_last_purchase.

För total_spend blir Low: >384, Mid: 384-1212 , High: 1212+ 
För total_orders blir Low: 1, Mid: 2-4, High: 5+

(Nivåer satta vid 33e och 67e percentilen, exempelvis "High", är relativt 
till kundbasen, inte ett absolut högt belopp. De speglar relativa skillnader i kundbasen.)

Vi applicerar sedan dessa nivåer för att tilldela kunderna poäng, 1 ,2 och 3, där högre är bättre.

In [118]:
# RFM-scoring
customer_scoring = db.sql("""
                          SELECT
                          Customerid,
                          total_spend,
                          total_orders,
                          days_since_last_purchase,

                          CASE 
                            WHEN total_spend >= 1212 THEN 3
                            WHEN total_spend >= 384 THEN 2
                            ELSE 1 
                          END as monetary_score,
                          
                          CASE
                            WHEN total_orders >= 5 THEN 3
                            WHEN total_orders >= 2 THEN 2
                            ELSE 1 
                          END as frequency_score,

                          CASE
                            WHEN days_since_last_purchase <= 39 THEN 3 
                            WHEN days_since_last_purchase <= 89 THEN 2
                            ELSE 1
                          END as recency_score

                          FROM customerdim
                          WHERE (customerid NOT IN (
                              SELECT DISTINCT customerid 
                              FROM staging.top_anomalies 
                              WHERE anomaly_type = 'anomaly_customer'
                              ) OR customerid IS NULL) 
                          """).df()
customer_scoring[['monetary_score','recency_score','frequency_score']].apply(lambda col: col.value_counts())

,monetary_score,recency_score,frequency_score
1,1464,1480,1526
2,1474,956,1728
3,1431,1933,1115


In [119]:
# Ser över distribution av scoring
print(customer_scoring[['monetary_score']].value_counts().sort_index())
print(customer_scoring[['frequency_score']].value_counts().sort_index())
print(customer_scoring[['recency_score']].value_counts().sort_index())
# Det lägre antalet i recency_score: 2 kan tolkas som övergångsfas mellan aktiv och inaktiv, där kunder sällan befinner sig en längre tid.

monetary_score
1                 1464
2                 1474
3                 1431
Name: count, dtype: int64
frequency_score
1                  1526
2                  1728
3                  1115
Name: count, dtype: int64
recency_score
1                1480
2                 956
3                1933
Name: count, dtype: int64


Vi delar sedan upp kunderna i sex grupper. Champions, Loyal, Big spender, New, At Risk och Opportunity. Kunderna kvalificerar sig i dessa grupper beroende på de poäng vi tilldelat dem.

    Champions:   R = 3    F = 3    M = 3   (Motivering?)
    Big Spender: R = 1-3  F = 1-3  M = 3   (Motivering?)
    Loyal:       R = 2-3  F = 3    M = 1-3 (Motivering?) 
    At Risk:     R = 1-2    F = 1-3  M = 2-3 (Signalerar kunder vi kan rikta retention-kampanjer mot)
    New:         R = 3    F = 1    M = 1-3 (Motivering?)
    Opportunity: R = 2    F = 2    M = 1-3   (Motivering?) 

I och med att "Big Spender" och "Loyal" kan överlapa så kommer kunder som kvalificerar sig som "Big Spender" att hamna i det segmentet. Denna prioritering baseras på de olika typer av insatser som är bäst lämpade för de två segmenten. Big Spenders som driver intäkter på kort sikt motiverar då potentiellt kortsiktiga och flexibla insatser, medan Loyal per design är ett långsiktigt relationsbygge. 

In [120]:
# Segmentlogik och segmentering
customer_segment = db.sql("""
                          WITH customer_scoring as(
                          SELECT
                          *,

                          CASE 
                            WHEN total_spend >= 1212 THEN 3
                            WHEN total_spend >= 384 THEN 2
                            ELSE 1 
                          END as m_score,
                          
                          CASE
                            WHEN total_orders >= 5 THEN 3
                            WHEN total_orders >= 2 THEN 2
                            ELSE 1 
                          END as f_score,

                          CASE
                            WHEN days_since_last_purchase <= 39 THEN 3 
                            WHEN days_since_last_purchase <= 89 THEN 2
                            ELSE 1
                          END as r_score

                          FROM customerdim
                          WHERE (customerid NOT IN (
                              SELECT DISTINCT customerid 
                              FROM staging.top_anomalies 
                              WHERE anomaly_type = 'anomaly_customer'
                              ) OR customerid IS NULL) 
                          )

                          SELECT
                          *,

                          CASE
                            WHEN r_score = 3 AND f_score = 3 AND m_score = 3 THEN 'champion'
 
                            WHEN r_score >= 1 AND f_score >= 1 AND m_score = 3 THEN 'big_spender'
                    
                            WHEN r_score >= 2 AND f_score = 3 AND m_score >= 1 THEN 'loyal'
                        
                            WHEN r_score <= 2 AND f_score >= 1 AND m_score >= 2 THEN 'at_risk'
                         
                            WHEN r_score = 3 AND f_score = 1 AND m_score >= 1 THEN 'new'
                       
                            WHEN r_score >= 2 AND f_score = 2 AND m_score >= 1 THEN 'opportunity'
                          
                            ELSE 'inactive_other'
                          
                          END as segment

                          FROM customer_scoring
                          
                          """).df()

In [121]:
# Ser över de kunder vars poäng inte ledde till segmentering. Gör även eventuella justeringar i cutoffs.
print(customer_segment[customer_segment['segment'].isna()][['r_score','f_score','m_score']].value_counts())
# Ser sedan över distribution av segment.
print(customer_segment.groupby('segment').size().sort_values(ascending=False))
# Lågt antal i "loyal" på grund av överlapp med big_spender som har prio. Champions klassas tekniskt sett också som loyal.

Series([], Name: count, dtype: int64)
segment
inactive_other    1037
at_risk            857
champion           781
big_spender        650
opportunity        642
new                280
loyal              122
dtype: int64


In [122]:
db.register('customer_segment_tbl', customer_segment)

In [ ]:
# Segmentöversikt intäktsmått
db.sql("""SELECT 
       COUNT(*) as customers,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) as pct_customer,
       segment,
       ROUND(SUM(total_spend), 2) as spending,
       ROUND(SUM(total_spend) * 100.0 / SUM(SUM(total_spend)) OVER (), 2) as spending_share,
       ROUND(SUM(total_spend) / COUNT(*)) as avg_customer_spend
       FROM customer_segment_tbl 
       GROUP BY segment
       ORDER BY spending desc
       """)

┌───────────┬──────────────┬────────────────┬────────────┬────────────────┬────────────────────┐
│ customers │ pct_customer │    segment     │  spending  │ spending_share │ avg_customer_spend │
│   int64   │    double    │    varchar     │   double   │     double     │       double       │
├───────────┼──────────────┼────────────────┼────────────┼────────────────┼────────────────────┤
│       781 │         17.9 │ champion       │ 5621237.39 │          65.35 │             7197.0 │
│       650 │         14.9 │ big_spender    │ 1624520.52 │          18.89 │             2499.0 │
│       857 │         19.6 │ at_risk        │  578957.21 │           6.73 │              676.0 │
│       642 │         14.7 │ opportunity    │  373676.33 │           4.34 │              582.0 │
│      1037 │         23.7 │ inactive_other │  210920.32 │           2.45 │              203.0 │
│       122 │          2.8 │ loyal          │  111294.98 │           1.29 │              912.0 │
│       280 │          6.4 │ n

Segmentöversikten ger oss en klar bild över vilka kunder som faktiskt driver intäkter.
Intäkterna är tydligt koncentrerade till en liten del av kundbasen. Champions (17,9% av kunderna) står för 65,35% av omsättningen, och tillsammans med “Big Spenders” (14,9%) genererar de 84,2% av intäkterna från registrerade kunder.

Att en liten del av kundbasen står för en betydande del av intäkterna skapar både möjlighet för bättre riktade retention-insatser, men utgör också en risk på grund av intäktskoncentrationen.

work in progress